# Hafta 6 · Çok Kübitli Devreler: CNOT, Dolanıklık ve Işınlanma
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~50 dk · **Ortam:** Google Colab

Bu hafta tek kübitten çıkıp **kübitleri birbirine bağlayan kapılarla** çalışıyoruz. Önce her kapıyı otomatik test eden küçük bir `truth_table()` aracı yazacağız; sonra bu araçla CNOT, CZ, SWAP ve Toffoli'yi doğrulayacak, kuantum kapılarıyla bir **toplayıcı devresi** kuracak, Bell ve GHZ durumlarının korelasyonlarını ölçecek ve son olarak **ışınlanma** ile **süper yoğun kodlama** protokollerini çalıştıracağız.

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum ve yardımcı fonksiyonlar | 3 dk |
| A | CNOT: matris, bit sırası, `truth_table()` test aracı | 8 dk |
| B | CZ, SWAP, Toffoli, kontrollü-U, faz geri tepmesi | 8 dk |
| C | Bell ve GHZ üreteci, ZZ/XX korelasyon ölçümü | 8 dk |
| D | Tersinir mantık: yarım toplayıcı, tam toplayıcı, uncompute | 8 dk |
| E | Işınlanma: adım adım, dinamik devre, ertelenmiş ölçüm, fidelity | 12 dk |
| F | Süper yoğun kodlama | 3 dk |
| G | Alıştırmalar (8 adet, `assert` ile) | ödev |

**Bit sırası kuralı (bütün ders boyunca):** Qiskit sırası, yani sonuç stringinde **q₀ en sağdadır**. `'10'` → q₁ = 1, q₀ = 0.

## 0 · Kurulum

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import RYGate, HGate, XGate
from qiskit.quantum_info import Statevector, Operator, DensityMatrix, partial_trace, state_fidelity, random_statevector
from qiskit_aer import AerSimulator

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6"
aer = AerSimulator(seed_simulator=2026)

def run_counts(qc, shots=2000, seed=None):
    """Devreyi Aer'de çalıştırıp sayımları döndürür."""
    sim = AerSimulator(seed_simulator=seed) if seed is not None else aer
    return sim.run(transpile(qc, sim), shots=shots).result().get_counts()

def show_amps(qc_or_sv, title=""):
    """Durum vektörünü 'bit dizisi: genlik' tablosu olarak yazdırır (sıfırları atlar)."""
    sv = qc_or_sv if isinstance(qc_or_sv, Statevector) else Statevector(qc_or_sv)
    n = sv.num_qubits
    if title: print(title)
    for i, a in enumerate(sv.data):
        if abs(a) > 1e-9:
            print(f"  |{i:0{n}b}⟩ : {np.real_if_close(np.round(a, 3))}")

def bar_counts(counts, title=""):
    keys = sorted(counts)
    plt.figure(figsize=(max(3, 0.7*len(keys)), 2.6))
    plt.bar(keys, [counts[k] for k in keys], color=NAVY); plt.title(title, color=NAVY); plt.show()

def state_to_bloch(amps):
    a, b = complex(amps[0]), complex(amps[1])
    n = np.sqrt(abs(a)**2 + abs(b)**2); a, b = a/n, b/n
    return np.array([2*(np.conj(a)*b).real, 2*(np.conj(a)*b).imag, abs(a)**2 - abs(b)**2])

def plot_bloch(amps_list, titles=None):
    """Kübit durumlarını yan yana Bloch küresinde çizer (1. haftadaki fonksiyon)."""
    if not isinstance(amps_list, (list, tuple)) or np.isscalar(amps_list[0]): amps_list = [amps_list]
    k = len(amps_list); fig = plt.figure(figsize=(3.6*k, 3.8))
    for i, amps in enumerate(amps_list):
        ax = fig.add_subplot(1, k, i+1, projection="3d"); ax.set_box_aspect((1,1,1), zoom=1.3); ax.computed_zorder = False
        u, v = np.linspace(0, 2*np.pi, 50), np.linspace(0, np.pi, 25)
        ax.plot_surface(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)), np.outer(np.ones_like(u), np.cos(v)),
                        color="#EEF2F8", alpha=0.25, linewidth=0, shade=False)
        t = np.linspace(0, 2*np.pi, 200)
        ax.plot(np.cos(t), np.sin(t), 0, color=GRAY, lw=0.8); ax.plot(np.cos(t), 0*t, np.sin(t), color=GRAY, lw=0.5); ax.plot(0*t, np.cos(t), np.sin(t), color=GRAY, lw=0.5)
        for d in [(1,0,0), (0,1,0), (0,0,1)]: ax.plot([-d[0], d[0]], [-d[1], d[1]], [-d[2], d[2]], color=GRAY, lw=0.7, ls="--")
        for p, s in [((0,0,1.22),"|0⟩ (z)"), ((0,0,-1.25),"|1⟩"), ((1.42,0,0),"|+⟩ (x)"), ((-1.32,0,0),"|−⟩"), ((0,1.32,0),"|+i⟩ (y)"), ((0,-1.32,0),"|−i⟩")]:
            ax.text(*p, s, ha="center", va="center", fontsize=9.5, color=NAVY)
        x, y, z = state_to_bloch(amps) if len(amps) == 2 else amps   # 3 elemanlı ise doğrudan Bloch vektörü
        ax.plot([0, x], [0, y], [0, z], color=BLUE, lw=3); ax.scatter([x], [y], [z], color=BLUE, s=60, depthshade=False)
        ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_zlim(-1.1, 1.1); ax.view_init(elev=18, azim=30); ax.set_axis_off()
        if titles: ax.set_title(titles[i], fontsize=11, color=NAVY)
    plt.show()
print("hazır")

---
## A · CNOT: kontrollü NOT
Klasik girdide CNOT basit bir `if`'tir: **kontrol 1 ise hedefi çevir** → `t = t XOR c`. Kontrol biti hiç değişmez.

| c | t (önce) | t (sonra) = t ⊕ c |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

⚠️ **4×4 matris, hangi kübitin kontrol olduğuna bağlıdır.** Qiskit sırasında (indeks = q₁q₀):
- `cx(0, 1)` (q₀ kontrol): |01⟩ ↔ |11⟩ takas eder
- `cx(1, 0)` (q₁ kontrol): |10⟩ ↔ |11⟩ takas eder — ders kitaplarındaki "CNOT matrisi" genellikle budur!

In [ ]:
qc01 = QuantumCircuit(2); qc01.cx(0, 1)
qc10 = QuantumCircuit(2); qc10.cx(1, 0)
print("cx(0,1) matrisi (q0 kontrol):\n", Operator(qc01).data.real)
print("cx(1,0) matrisi (q1 kontrol):\n", Operator(qc10).data.real)
qc01.draw("mpl")

In [ ]:
# Klasik XOR benzetmesi: bit işlemleriyle CNOT'un indeks permütasyonu (2. haftadaki MiniSim.apply_cx)
def cx_perm(i, c, t):
    return i ^ (1 << t) if (i >> c) & 1 else i

for i in range(4):
    print(f"|{i:02b}⟩ --cx(0,1)--> |{cx_perm(i, 0, 1):02b}⟩      --cx(1,0)--> |{cx_perm(i, 1, 0):02b}⟩")

### `truth_table()`: kuantum devreleri için birim test aracı
Bir devreyi **tüm klasik girdilerle** (000…, 001…, …) besleyip çıktıyı okur. Klasik girdide CNOT, SWAP, Toffoli gibi kapılar **deterministik** çalışır (olasılık 1). Çıktı kesin değilse (ör. H içeren devre) tablo bunu `?` ile işaretler.

In [ ]:
def truth_table(circuit, show=True):
    """Devrenin tüm klasik girdiler için çıktısını döndürür: {'girdi': 'çıktı'}.
    Bit stringleri Qiskit sırasındadır (q0 en sağda). Çıktı kesin değilse değer '?'."""
    n = circuit.num_qubits
    table = {}
    for i in range(2**n):
        bits = format(i, f"0{n}b")
        prep = QuantumCircuit(n)
        for q in range(n):
            if (i >> q) & 1:
                prep.x(q)                          # girdiyi hazırla
        probs = Statevector(prep.compose(circuit)).probabilities()
        j = int(np.argmax(probs))
        table[bits] = format(j, f"0{n}b") if probs[j] > 1 - 1e-9 else "?"
    if show:
        print("girdi -> çıktı")
        for k, v in table.items():
            print(f"  {k}  ->  {v}")
    return table

tt = truth_table(qc01)

### Süperpozisyonlu girdide CNOT: dolanıklık doğuyor
Kontrol kübiti |+⟩ ise CNOT **her iki dalı birden** çalıştırır: 00 dalında hiçbir şey olmaz, 01 dalında hedef çevrilir → 11. Sonuç bir **Bell durumu**dur.

In [ ]:
qc = QuantumCircuit(2)
show_amps(qc, "1) başlangıç:")
qc.h(0);     show_amps(qc, "2) h(0) sonrası:")
qc.cx(0, 1); show_amps(qc, "3) cx(0,1) sonrası:")

def is_product_state(sv):
    """2 kübitli durum çarpım durumu mu? (2. hafta: 2x2 matrisin rankı)"""
    return np.linalg.matrix_rank(np.asarray(sv).reshape(2, 2), tol=1e-9) == 1
print("Çarpım durumu mu?", is_product_state(Statevector(qc).data), " -> dolanık")

# Dolanık kübitin 'kendi' Bloch vektörü: kısmi iz ile tek kübitlik yoğunluk matrisi
rho0 = partial_trace(DensityMatrix(qc), [1]).data
print("q0'ın tek başına yoğunluk matrisi:\n", rho0.real, "\n-> Bloch vektörü boyu 0: q0 tek başına tamamen rastgele")

---
## B · Diğer çok kübitli kapılar
| Kapı | Qiskit | Klasik girdide etkisi | Özel not |
|---|---|---|---|
| CZ | `qc.cz(a, b)` | Bitler değişmez; |11⟩ genliği −1 ile çarpılır | Simetrik: `cz(0,1) == cz(1,0)` |
| SWAP | `qc.swap(a, b)` | İki kübiti takas eder | 3 CNOT = XOR takas hilesi |
| Toffoli | `qc.ccx(a, b, t)` | `t ^= (a AND b)` | Tersinir AND |
| Kontrollü-U | `U.control(k)` | Kontroller 1 ise U uygulanır | `ch`, `cry`, `mcx` ... |

In [ ]:
cz01 = QuantumCircuit(2); cz01.cz(0, 1)
cz10 = QuantumCircuit(2); cz10.cz(1, 0)
print("CZ matrisi (köşegen):", np.diag(Operator(cz01).data).real)
print("cz(0,1) == cz(1,0) ?", Operator(cz01).equiv(Operator(cz10)))

hxh = QuantumCircuit(2); hxh.h(1); hxh.cx(0, 1); hxh.h(1)
print("H·CX·H (hedefte) == CZ ?", Operator(hxh).equiv(Operator(cz01)))
print("CZ klasik girdide bitleri değiştirmez:"); truth_table(cz01)

In [ ]:
sw = QuantumCircuit(2); sw.swap(0, 1)
sw3 = QuantumCircuit(2); sw3.cx(0, 1); sw3.cx(1, 0); sw3.cx(0, 1)
print("3 CNOT == SWAP ?", Operator(sw3).equiv(Operator(sw)))

# Klasik XOR takas: a ^= b; b ^= a; a ^= b
a, b = 1, 0
b ^= a; a ^= b; b ^= a
print("XOR takas sonrası a, b =", a, b)

# Süperpozisyonlu bir durumu da takas eder
qc = QuantumCircuit(2); qc.ry(1.2, 0)            # q0 = Ry(1.2)|0>, q1 = |0>
print("önce :", Statevector(qc).data.round(3))
qc.compose(sw3, inplace=True)
print("sonra:", Statevector(qc).data.round(3), " (genlik 01'den 10'a taşındı)")
sw3.draw("mpl")

In [ ]:
tof = QuantumCircuit(3); tof.ccx(0, 1, 2)
tt = truth_table(tof)
print("Toffoli matrisi permütasyon mu? (her satırda tek 1):", np.allclose(np.abs(Operator(tof).data).sum(1), 1))
tof.draw("mpl")

### Genel kontrollü-U: `.control()`
Her kapıyı kontrollü hâle getirebiliriz. Kontrollü-U matrisi blok köşegendir: kontrol 0 iken I, 1 iken U.

In [ ]:
cry = QuantumCircuit(2); cry.append(RYGate(0.8).control(1), [0, 1])      # kontrol q0, hedef q1
cry_builtin = QuantumCircuit(2); cry_builtin.cry(0.8, 0, 1)
print(".control() == qc.cry ?", Operator(cry).equiv(Operator(cry_builtin)))

cch = QuantumCircuit(3); cch.append(HGate().control(2), [0, 1, 2])         # iki kontrollü H
print("iki kontrollü H, girdi 011 (q0=q1=1):")
p = QuantumCircuit(3); p.x([0, 1]); show_amps(p.compose(cch))

mcx = QuantumCircuit(4); mcx.mcx([0, 1, 2], 3)                               # 3 kontrollü X
print("mcx: yalnız 0111 girdisinde hedef çevrilir ->", truth_table(mcx, show=False)["0111"])
cch.draw("mpl")

### Faz geri tepmesi (phase kickback) — 8. haftaya köprü
Hedef kübit |−⟩ ise, CNOT hedefi değiştirmez (|−⟩, X'in "özdurumu"dur: X|−⟩ = −|−⟩). Ortaya çıkan −1 işareti **kontrol kübitine** yazılır. Kontrol |+⟩ idiyse |−⟩'ye döner. 8. haftadaki Deutsch-Jozsa ve Bernstein-Vazirani algoritmaları tamamen bu hileye dayanır.

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)                    # kontrol: |+>
qc.x(1); qc.h(1)           # hedef:   |->
before = partial_trace(DensityMatrix(qc), [1]).data
qc.cx(0, 1)
after = partial_trace(DensityMatrix(qc), [1]).data
def bloch_from_rho(rho): return np.array([2*rho[0,1].real, -2*rho[0,1].imag, (rho[0,0]-rho[1,1]).real])
print("q0 Bloch önce:", bloch_from_rho(before).round(3), " sonra:", bloch_from_rho(after).round(3))
qc.h(0); qc.h(1)
m = qc.copy(); m.measure_all()
print("h ile geri çevirip ölçünce:", run_counts(m, 1000, seed=1), "-> q0 kesin 1 (hedef q1 de kesin 1)")
plot_bloch([bloch_from_rho(before), bloch_from_rho(after)], ["q0: CNOT öncesi |+⟩", "q0: CNOT sonrası |−⟩"])

---
## C · Bell ve GHZ durumları; korelasyon ölçümü
| Bell durumu | Girdi | Devre | Qiskit sırasında genlikler (00, 01, 10, 11) |
|---|---|---|---|
| Φ⁺ | \|00⟩ | h(0), cx(0,1) | [½√2, 0, 0, ½√2] |
| Φ⁻ | \|01⟩ (x(0)) | x(0), h(0), cx(0,1) | [½√2, 0, 0, −½√2] |
| Ψ⁺ | \|10⟩ (x(1)) | x(1), h(0), cx(0,1) | [0, ½√2, ½√2, 0] |
| Ψ⁻ | \|11⟩ (x(0), x(1)) | x(0), x(1), h(0), cx(0,1) | [0, −½√2, ½√2, 0] |

In [ ]:
def bell_circuit(kind="phi+"):
    qc = QuantumCircuit(2)
    if kind in ("phi-", "psi-"): qc.x(0)
    if kind in ("psi+", "psi-"): qc.x(1)
    qc.h(0); qc.cx(0, 1)
    return qc

for k in ["phi+", "phi-", "psi+", "psi-"]:
    sv = Statevector(bell_circuit(k))
    m = bell_circuit(k); m.measure_all()
    print(f"{k:5s} genlikler: {np.real_if_close(sv.data).round(3)}   sayımlar: {run_counts(m, 1000, seed=3)}")

In [ ]:
def ghz_chain(n):
    qc = QuantumCircuit(n); qc.h(0)
    for k in range(n - 1): qc.cx(k, k + 1)
    return qc

for n in [3, 4, 5]:
    qc = ghz_chain(n); m = qc.copy(); m.measure_all()
    print(f"GHZ-{n}: derinlik {qc.depth()}, sayımlar {run_counts(m, 1000, seed=n)}")
ghz_chain(4).draw("mpl")

### Dolanıklığı istatistikle görmek: ZZ ve XX bazında eşleşme oranı
**Fizik yok, sadece istatistik.** İki kübiti ölçüp "iki bit aynı mı?" diye sayıyoruz:
- **ZZ bazı:** doğrudan ölçüm.
- **XX bazı:** ölçümden önce her iki kübite H (3. haftada gördüğümüz baz değiştirme).

"Yazı-tura atıp iki kutuya da aynı sonucu koymak" (klasik korelasyon) ZZ'de %100 eşleşme verir ama XX'te %50'ye düşer. Bell durumu **her iki bazda da** %100 eşleşir. Hiçbir çarpım durumu bunu yapamaz.

In [ ]:
def match_rate(counts):
    """Sayımlardan iki bitin aynı çıkma oranı (yalnızca ilk iki bite bakar)."""
    tot = sum(counts.values())
    return sum(v for k, v in counts.items() if k.replace(" ", "")[-1] == k.replace(" ", "")[-2]) / tot

def measure_in(prep, basis, shots=4000, seed=5):
    qc = prep.copy()
    if basis == "XX": qc.h([0, 1])
    cr = ClassicalRegister(2, "m"); qc.add_register(cr); qc.measure([0, 1], cr)
    return run_counts(qc, shots, seed)

cands = {"Bell Φ⁺": bell_circuit("phi+"), "|00⟩": QuantumCircuit(2)}
q = QuantumCircuit(2); q.h([0, 1]); cands["|+⟩|+⟩"] = q
q = QuantumCircuit(3); q.h(2); q.cx(2, 0); q.cx(2, 1); cands["klasik yazı-tura"] = q   # gizli q2 yazı-tura atar
print(f"{'durum':18s} {'ZZ':>6s} {'XX':>6s}")
for name, prep in cands.items():
    print(f"{name:18s} {match_rate(measure_in(prep, 'ZZ')):6.3f} {match_rate(measure_in(prep, 'XX')):6.3f}")

---
## D · Tersinir klasik mantık: kuantum kapılarıyla toplama
2\. haftada gördük: her kapı üniterdir → **geri alınabilir**. Klasik AND geri alınamaz (çıktı 0 ise girdi?). Çözüm: girdileri koru, sonucu ayrı bir kübite XOR'la.

| Klasik | Tersinir karşılığı | Qiskit |
|---|---|---|
| NOT | X | `qc.x(a)` |
| XOR (b ← a ⊕ b) | CNOT | `qc.cx(a, b)` |
| AND (t ← t ⊕ ab) | Toffoli | `qc.ccx(a, b, t)` |

**Yarım toplayıcı:** sum = a XOR b, carry = a AND b. Sıra önemli: önce carry (Toffoli, b bozulmadan), sonra sum (CNOT, b'nin üzerine yazar).

In [ ]:
def half_adder():
    """q0 = a, q1 = b (sonunda sum), q2 = carry (0 ile başlar)."""
    qc = QuantumCircuit(3, name="HA")
    qc.ccx(0, 1, 2)        # carry = a AND b
    qc.cx(0, 1)            # b <- a XOR b = sum
    return qc

ha = half_adder()
tt = truth_table(ha, show=False)
print(" a b | sum carry")
for a, b in product([0, 1], repeat=2):
    out = tt[f"0{b}{a}"]                  # girdi stringi q2 q1 q0 = 0 b a
    print(f" {a} {b} |  {out[1]}    {out[0]}")
    assert int(out[1]) == a ^ b and int(out[0]) == a & b
print("Yarım toplayıcı tüm girdilerde doğru ✓")
ha.draw("mpl")

### Tam toplayıcı (bonus)
Girdi: a, b, cin. Çıktı: sum, cout. 2 Toffoli + 3 CNOT ile; son CNOT b'yi eski hâline döndürür (temizlik).

In [ ]:
def full_adder():
    """q0=a, q1=b, q2=cin (sonunda sum), q3=cout (0 ile başlar). Sonunda a ve b korunur."""
    qc = QuantumCircuit(4, name="FA")
    qc.ccx(0, 1, 3)        # cout ^= a·b
    qc.cx(0, 1)            # b = a⊕b
    qc.ccx(1, 2, 3)        # cout ^= (a⊕b)·cin
    qc.cx(1, 2)            # cin = a⊕b⊕cin = sum
    qc.cx(0, 1)            # b'yi geri al
    return qc

tt = truth_table(full_adder(), show=False)
for a, b, c in product([0, 1], repeat=3):
    out = tt[f"0{c}{b}{a}"]
    s_, co = int(out[1]), int(out[0])
    assert s_ + 2*co == a + b + c and out[2:] == f"{b}{a}"
print("Tam toplayıcı 8 girdinin hepsinde a+b+cin = sum + 2·cout ✓")
full_adder().draw("mpl")

### Uncompute: yardımcı kübiti temizlemek
Ara sonucu (yardımcı / ancilla) sonucu kopyaladıktan sonra aynı kapıyı **tekrar uygulayarak** sıfırlarız. Temizlenmemiş yardımcı kübit, süperpozisyonda girdilerle dolanık kalır ve sonraki adımları bozar (8–9. haftalarda oracle yazarken çok önemli).

In [ ]:
def and_with_uncompute():
    qc = QuantumCircuit(4)          # q0,q1 = x ; q2 = yardımcı ; q3 = çıktı
    qc.ccx(0, 1, 2)                 # hesapla
    qc.cx(2, 3)                     # sonucu kopyala
    qc.ccx(0, 1, 2)                 # geri al (uncompute)
    return qc

tt = truth_table(and_with_uncompute(), show=False)
print("yardımcı kübit (q2) her girdide 0'a döndü mü?", all(v[1] == "0" for k, v in tt.items() if k[1] == "0"))

# Süperpozisyonda: girdi H ile tüm kombinasyonlarda. Temizlenmiş ve temizlenmemiş sürümü karşılaştır
for name, clean in [("uncompute YOK", False), ("uncompute VAR", True)]:
    qc = QuantumCircuit(4); qc.h([0, 1]); qc.ccx(0, 1, 2); qc.cx(2, 3)
    if clean: qc.ccx(0, 1, 2)
    rho_anc = partial_trace(DensityMatrix(qc), [0, 1, 3]).data
    print(f"{name}: yardımcı kübitte P(1) = {rho_anc[1,1].real:.2f}")
and_with_uncompute().draw("mpl")

---
## E · Işınlanma (teleportation)
**Yazılım gözüyle:** Önceden paylaşılmış bir Bell çifti + **2 klasik bit** ile bilinmeyen bir kübit durumunu (q₀) başka bir kübite (q₂) **taşırız**. Durum kopyalanmaz: q₀ ölçülür ve bozulur (klonlanamazlık → `std::move` gibi).

1. q₁–q₂ arasında Bell çifti: `h(1); cx(1, 2)` (q₂ alıcıdadır)
2. Gönderen: `cx(0, 1); h(0)`, sonra q₀ → m₀, q₁ → m₁ ölçülür
3. 2 klasik bit gönderilir; alıcı düzeltir: **m₁ = 1 → X**, **m₀ = 1 → Z**

| m₀ m₁ (ölçüm sırası) | Qiskit sayım stringi `m1 m0` | Düzeltme |
|---|---|---|
| 0 0 | `0 0` | I |
| 0 1 | `1 0` | X |
| 1 0 | `0 1` | Z |
| 1 1 | `1 1` | Z·X (önce X, sonra Z) |

In [ ]:
# Adım adım genlikler (ψ = [0.6, 0.8])
qc = QuantumCircuit(3); qc.initialize([0.6, 0.8], 0)
show_amps(qc, "0) başlangıç ψ ⊗ |00⟩:")
qc.h(1); qc.cx(1, 2);  show_amps(qc, "1) Bell çifti (q1,q2):")
qc.cx(0, 1);           show_amps(qc, "2) cx(0,1):")
qc.h(0);               show_amps(qc, "3) h(0):")
print("\nHer (m1 m0) sonucu için q2'nin koşullu durumu:")
sv = Statevector(qc).data
for m1, m0 in product([0, 1], repeat=2):
    v = np.array([sv[m0 + 2*m1 + 4*k] for k in (0, 1)]); v = v / np.linalg.norm(v)
    print(f"  m1 m0 = {m1}{m0}: q2 = {np.real_if_close(v).round(3)}")

### Sürüm 1 · Dinamik devre (Qiskit 2.x `if_test`)
Ölçüm sonucu devrenin ortasında okunur ve sonraki kapıyı koşullar. Qiskit 2.x'te eski `c_if` kaldırıldı; yerine `with qc.if_test((clbit, 1)):` kullanılır.

In [ ]:
def teleport_dynamic(psi):
    q = QuantumRegister(3, "q"); m0 = ClassicalRegister(1, "m0"); m1 = ClassicalRegister(1, "m1")
    qc = QuantumCircuit(q, m0, m1)
    qc.initialize(psi, 0)
    qc.h(1); qc.cx(1, 2)                  # paylaşılan Bell çifti
    qc.cx(0, 1); qc.h(0)                  # gönderen
    qc.measure(0, m0[0]); qc.measure(1, m1[0])
    with qc.if_test((m1[0], 1)):          # alıcı: düzeltmeler
        qc.x(2)
    with qc.if_test((m0[0], 1)):
        qc.z(2)
    return qc

psi = random_statevector(2, seed=42)
qc = teleport_dynamic(psi.data)
qc_save = qc.copy(); qc_save.save_density_matrix([2], label="rho")    # q2'nin yoğunluk matrisi (tüm shot ortalaması)
dm_sim = AerSimulator(method="density_matrix")
res = dm_sim.run(transpile(qc_save, dm_sim), shots=500, seed_simulator=7).result()
rho_q2 = res.data()["rho"]
print("ölçüm sayımları (m1 m0):", res.get_counts(), "-> dört sonuç da ~eşit olasılıklı")
print("fidelity(q2, ψ) =", round(state_fidelity(rho_q2, psi), 6))
qc.draw("mpl", fold=-1)

### Sürüm 2 · Ertelenmiş ölçüm (deferred measurement)
Klasik koşullu kapılar yerine **kuantum kontrollü** kapılar: `cx(1, 2)` ve `cz(0, 2)`. Ölçüm sona ertelenebilir (ya da hiç yapılmaz). Böylece tüm devre üniter olur ve `Statevector` ile doğrudan simüle edilir.

In [ ]:
def teleport_deferred(psi):
    qc = QuantumCircuit(3)
    qc.initialize(psi, 0)
    qc.h(1); qc.cx(1, 2)
    qc.cx(0, 1); qc.h(0)
    qc.cx(1, 2); qc.cz(0, 2)      # düzeltmeler (m1 -> X, m0 -> Z) kuantum kontrollü
    return qc

def fidelity_q2(qc, psi):
    rho = partial_trace(Statevector(qc), [0, 1])     # q0 ve q1'i "at", q2'yi tut
    return state_fidelity(rho, Statevector(psi))

fids = []
for seed in range(10):
    psi = random_statevector(2, seed=seed).data
    fids.append(fidelity_q2(teleport_deferred(psi), psi))
print("10 rastgele durumda fidelity:", np.round(fids, 6))
teleport_deferred([0.6, 0.8]).draw("mpl")

### Doğrulama 3 · Tomografi ile (yalnızca sayımlardan)
Gerçek donanımda yoğunluk matrisini göremeyiz. q₂'yi X, Y, Z bazlarında ölçüp Bloch vektörünü tahmin ederiz (3. hafta). Dinamik sürümde q₂'yi düzeltmeden **sonra** ölçüyoruz.

In [ ]:
def teleport_tomography(psi, shots=4000, seed=1):
    bloch = []
    for basis in "XYZ":
        qc = teleport_dynamic(psi)
        out = ClassicalRegister(1, "out"); qc.add_register(out)
        if basis == "X": qc.h(2)
        if basis == "Y": qc.sdg(2); qc.h(2)
        qc.measure(2, out[0])
        cnt = run_counts(qc, shots, seed)
        n0 = sum(v for k, v in cnt.items() if k.split(" ")[0] == "0")   # 'out' en soldaki register
        bloch.append((2*n0 - shots) / shots)
    return np.array(bloch)

psi = random_statevector(2, seed=42).data
b_est, b_true = teleport_tomography(psi), state_to_bloch(psi)
F_est = 0.5 * (1 + np.dot(b_est, b_true))            # saf hedef durum için fidelity = (1 + r·s)/2
print("gerçek Bloch:", b_true.round(3), " tahmin:", b_est.round(3), " tahmini fidelity ≈", round(F_est, 3))
plot_bloch([psi, b_est], ["gönderilen ψ (q0)", "alınan (q2), tomografi"])

---
## F · Süper yoğun kodlama (superdense coding): 1 kübitle 2 bit
Işınlanmanın tersi: önceden paylaşılmış Bell çifti varsa, gönderen **tek bir kübit** yollayarak **2 klasik bit** iletir.
- Gönderen q₀'a kodlar: bit₀ (sağdaki) = 1 → **Z**, bit₁ (soldaki) = 1 → **X**
- Alıcı çözer: `cx(0, 1); h(0)` ve ölçer → sonuç stringi mesajın kendisidir.

In [ ]:
def superdense(msg):
    qc = QuantumCircuit(2)
    qc.h(0); qc.cx(0, 1)              # paylaşılmış Bell çifti
    if msg[1] == "1": qc.z(0)         # sağdaki bit
    if msg[0] == "1": qc.x(0)         # soldaki bit
    qc.cx(0, 1); qc.h(0)              # alıcı çözer
    qc.measure_all()
    return qc

for msg in ["00", "01", "10", "11"]:
    print(msg, "->", run_counts(superdense(msg), 500, seed=1))
superdense("11").draw("mpl")

---
## G · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur.

### Alıştırma 1 · CNOT matrisini elle kurmak
`cnot_matrix(c, t, n)` fonksiyonu n kübitlik CNOT'un 2ⁿ×2ⁿ permütasyon matrisini döndürsün (Qiskit sırası). İpucu: sütun `i` → satır `i ^ (1<<t)` (kontrol biti 1 ise).

In [ ]:
def cnot_matrix(c, t, n=2):
    M = np.zeros((2**n, 2**n))
    # TODO
    return M

for (c, t, n) in [(0, 1, 2), (1, 0, 2), (0, 2, 3), (2, 1, 3)]:
    ref = QuantumCircuit(n); ref.cx(c, t)
    assert np.allclose(cnot_matrix(c, t, n), Operator(ref).data), (c, t, n)
print("Alıştırma 1 ✓")

### Alıştırma 2 · CZ'yi CNOT ile, CNOT'u CZ ile
`cz_from_cx()` yalnızca H ve CX kullanarak CZ'yi, `cx_from_cz()` yalnızca H ve CZ kullanarak `cx(0,1)`'i kursun.

In [ ]:
def cz_from_cx():
    qc = QuantumCircuit(2)
    # TODO
    return qc

def cx_from_cz():
    qc = QuantumCircuit(2)
    # TODO
    return qc

ref_cz = QuantumCircuit(2); ref_cz.cz(0, 1)
ref_cx = QuantumCircuit(2); ref_cx.cx(0, 1)
assert set(cz_from_cx().count_ops()) <= {"h", "cx"} and Operator(cz_from_cx()).equiv(Operator(ref_cz))
assert set(cx_from_cz().count_ops()) <= {"h", "cz"} and Operator(cx_from_cz()).equiv(Operator(ref_cx))
print("Alıştırma 2 ✓")

### Alıştırma 3 · Bell durumu tanıyıcı
`identify_bell(qc)` verilen 2 kübitlik devrenin hangi Bell durumunu ürettiğini (`"phi+"`, `"phi-"`, `"psi+"`, `"psi-"`) ya da hiçbirini (`None`) döndürsün. İpucu: `Statevector.equiv()` global fazı yok sayar.

In [ ]:
def identify_bell(qc):
    # TODO
    pass

q = QuantumCircuit(2); q.h(1); q.cx(1, 0); q.z(1)
assert identify_bell(q) == "phi-"
assert identify_bell(bell_circuit("psi-")) == "psi-"
q = QuantumCircuit(2); q.h(0); q.cx(0, 1); q.x(1)
assert identify_bell(q) == "psi+"
assert identify_bell(QuantumCircuit(2)) is None
print("Alıştırma 3 ✓")

### Alıştırma 4 · Sığ GHZ (logaritmik derinlik)
`ghz_log(n)`: GHZ durumunu **1 + ⌈log₂ n⌉** derinlikle hazırlasın. İpucu: her turda "hazır" kübit sayısı ikiye katlanır: `cx(k, k + have)`.

In [ ]:
def ghz_log(n):
    qc = QuantumCircuit(n)
    # TODO
    return qc

import math
for n in [2, 3, 5, 8, 11]:
    qc = ghz_log(n)
    assert Statevector(qc).equiv(Statevector(ghz_chain(n))), n
    assert qc.depth() <= 1 + math.ceil(math.log2(n)), (n, qc.depth())
print("Alıştırma 4 ✓", [ghz_log(n).depth() for n in [2, 3, 5, 8, 11]])

### Alıştırma 5 · Korelasyon tablosu
`correlations(prep)` bir 2 kübitlik hazırlık devresi için `(ZZ_eşleşme, XX_eşleşme)` çiftini döndürsün (`measure_in` ve `match_rate` kullanın, 4000 shot). Bell Φ⁺ ≈ (1, 1), Ψ⁻ ≈ (0, 0), |00⟩ ≈ (1, 0.5) olmalı.

In [ ]:
def correlations(prep):
    # TODO
    pass

zz, xx = correlations(bell_circuit("phi+")); assert zz > 0.99 and xx > 0.99
zz, xx = correlations(bell_circuit("psi-")); assert zz < 0.01 and xx < 0.01
zz, xx = correlations(QuantumCircuit(2));    assert zz > 0.99 and abs(xx - 0.5) < 0.05
print("Alıştırma 5 ✓")

### Alıştırma 6 · 2 bitlik toplayıcı
`adder2()`: a = (q1 q0), b = (q3 q2) iki bitlik sayılar; toplam 3 bit olarak **s = (q6 q5 q4)** kübitlerine yazılsın ve a, b korunsun. q7 serbest yardımcı kübit (sonunda 0 olmalı). Yalnızca X, CX, CCX kullanın. Tüm 16 girdi `truth_table` ile test edilir.

In [ ]:
def adder2():
    qc = QuantumCircuit(8)
    # TODO
    return qc

tt = truth_table(adder2(), show=False)
for a, b in product(range(4), repeat=2):
    inp = f"0000{b:02b}{a:02b}"
    out = tt[inp]
    assert out != "?" and out[0] == "0" and out[-4:] == inp[-4:], (a, b, out)
    assert int(out[1:4], 2) == a + b, (a, b, out)
assert set(adder2().count_ops()) <= {"x", "cx", "ccx"}
print("Alıştırma 6 ✓")

### Alıştırma 7 · Işınlanmayı test et
`teleport_fidelities(k)`: `teleport_deferred` ile k rastgele durum (seed = 100…100+k−1) ışınlayıp fidelity listesini döndürsün. Ayrıca `broken_teleport(psi)` sürümünde **Z düzeltmesini atlayın**: fidelity'nin bazı durumlarda belirgin biçimde düştüğünü gösterin.

In [ ]:
def teleport_fidelities(k=8):
    # TODO
    pass

def broken_teleport(psi):
    qc = QuantumCircuit(3)
    # TODO: teleport_deferred ile aynı, ama cz(0, 2) yok
    return qc

f = teleport_fidelities(8)
assert len(f) == 8 and min(f) > 0.9999
fb = [fidelity_q2(broken_teleport(random_statevector(2, seed=s).data), random_statevector(2, seed=s).data) for s in range(100, 108)]
print("bozuk sürüm:", np.round(fb, 3))
assert min(fb) < 0.9
print("Alıştırma 7 ✓")

### Alıştırma 8 · Süper yoğun kodlama ile metin göndermek
`send_text(text)`: her karakteri 8 bite çevirip (`format(ord(ch), '08b')`) 2'şer bitlik parçalar hâlinde `superdense` ile gönderin (her parça 1 shot), alıcı tarafta metni geri kurun. Kaç kübit gönderildiğini de döndürün.

In [ ]:
def send_text(text):
    received_bits, qubits_sent = "", 0
    # TODO
    decoded = "".join(chr(int(received_bits[i:i+8], 2)) for i in range(0, len(received_bits), 8))
    return decoded, qubits_sent

msg, nq = send_text("Qiskit")
print(msg, nq)
assert msg == "Qiskit" and nq == 24
print("Alıştırma 8 ✓")

---
### Haftanın özeti
- **CNOT** = kontrollü NOT = XOR: `t ^= c`. 4×4 matris hangi kübitin kontrol olduğuna bağlıdır (Qiskit sırası!).
- Süperpozisyonlu kontrol + CNOT → **dolanıklık** (çarpana ayrılamayan durum).
- **CZ** simetriktir, yalnız |11⟩'in işaretini çevirir; **SWAP** = 3 CNOT; **Toffoli** = tersinir AND; her kapı `.control()` ile kontrollü olur.
- **Faz geri tepmesi:** hedef |−⟩ iken faz kontrole yazılır (8. hafta).
- **Bell/GHZ**: tek tek rastgele, birlikte tam korelasyonlu; ZZ **ve** XX bazında eşleşme dolanıklığın istatistik imzasıdır.
- **Tersinir mantık**: X, CX, CCX ile toplayıcı; yardımcı kübitleri **uncompute** ile temizle.
- **Işınlanma**: Bell çifti + 2 klasik bit → durum taşınır (kopyalanmaz). Dinamik devre (`if_test`) ya da ertelenmiş ölçüm (`cx`, `cz`); fidelity = 1.
- **Süper yoğun kodlama**: Bell çifti + 1 kübit → 2 klasik bit.

**Gelecek hafta (Hafta 7):** Framework'ler ve donanıma erişim: aynı Bell/ışınlanma devresini Qiskit, Cirq, PennyLane ve Braket'te yazacak; simülatör ile gerçek (gürültülü) donanım sonuçlarını karşılaştıracağız.